# Build a Phased Migration Plan — Solution


## Phase 1: Days 1–30 (Foundation + Parallel Run)

| Task | Dependencies | Success Criteria |
|------|-------------|-----------------|
| Deploy Athena workgroups + Iceberg gold tables in production | Gold layer validated in dev | All 12 dashboard queries return results from Athena |
| Configure Athena ODBC connector for Tableau | Athena workgroups live | Tableau can connect and run test queries |
| Set up parallel validation: run all 12 dashboard queries against both Redshift and Athena daily | Both engines accessible | Automated comparison job running; results logged |
| Migrate 3 lowest-risk dashboards to Athena | Parallel validation showing <2% variance | 3 dashboards rendering from Athena; users confirmed |
| Set Redshift to read-only for migrated tables | Dashboards confirmed on Athena | No writes to migrated tables |
| Train 25 analysts on Athena SQL differences | Athena accessible | Training sessions delivered; 25 users can run queries |

**Phase 1 success:** 3 dashboards on Athena, parallel validation running, team trained.

## Phase 2: Days 31–60 (Complete Migration + Decommission)

| Task | Dependencies | Success Criteria |
|------|-------------|-----------------|
| Migrate remaining 9 dashboards to Athena | Phase 1 validation showing <2% variance across all queries | All 12 dashboards rendering from Athena |
| Redirect 3 Glue jobs from Redshift to write to S3/Iceberg only | Gold Iceberg tables accepting writes | Glue jobs writing to Iceberg; Redshift receives no new data |
| Run 14-day bake period with Redshift read-only (no writes) | All consumers on Athena | Zero Redshift queries from production consumers for 7+ consecutive days |
| Terminate Redshift cluster | 14-day bake period passed; snapshot taken | Cluster deleted; $6,400/mo saved |
| Ad-hoc analyst migration: all 25 users running queries on Athena | Training complete + dashboards migrated | Redshift query audit log shows zero queries |
| Finance month-end validation (avoid days 1–7 disruption) | Schedule termination for day 8+ of month | Finance confirms reporting unaffected |

**Phase 2 success:** Redshift terminated, all consumers on Athena, cost savings realized.

## Dual-Running Validation

**Approach:** Automated daily comparison job (Lambda + Athena)

1. Run each of the 12 dashboard SQL queries against **both** Redshift and Athena
2. Compare result sets: row counts, SUM/AVG of numeric columns, NULL counts
3. Log results to S3 with pass/fail status
4. Threshold: **<2% variance** = pass; **>2%** = fail + alert

**Duration:** Runs from Day 5 through Day 45 (covers both phases)

**What it catches:** Missing data in Iceberg tables, transformation differences, timezone issues, stale partitions

## Rollback Triggers

| Trigger Condition | Action | Recovery Time |
|-------------------|--------|---------------|
| Dashboard query latency >30s for 3+ consecutive hours | Reconnect affected Tableau dashboards to Redshift (connection string swap) | <30 minutes |
| Parallel validation shows >5% data variance on any financial metric | Pause migration; investigate silver/gold logic; revert Finance dashboards to Redshift | <1 hour |
| Athena service outage >15 minutes during business hours | Activate Redshift read-replica fallback (kept available until Day 45) | <15 minutes (DNS/connection swap) |

## Risk Register

**Phase 1 Risks:**

| Risk | Impact | Mitigation |
|------|--------|-----------|
| Athena ODBC connector causes Tableau rendering issues | Dashboards broken for users | Test all 12 dashboards in staging; migrate lowest-risk first; keep Redshift active |
| Analysts struggle with Athena SQL dialect differences | Queries fail; productivity drops | Provide cheat-sheet of Redshift→Athena SQL differences; office hours for first 2 weeks |
| Finance month-close conflict (days 1–7 blackout) | Cannot migrate Finance dashboards during close | Schedule Finance dashboard migration for days 8–25 only; validate with Finance team first |

**Phase 2 Risks:**

| Risk | Impact | Mitigation |
|------|--------|-----------|
| Undiscovered Redshift consumers surface after termination | Broken queries for unknown teams | Monitor Redshift audit log for 14 days; announce deprecation company-wide at Day 30; keep snapshot 30 days |
| Athena costs spike beyond budget with 25 analysts running ad-hoc queries | Cost overrun | Set per-query scan limits (workgroup: max 10GB/query); optimize gold tables with Z-order; alert at 80% of monthly budget |
| Glue job failure during Iceberg write causes stale gold tables | Dashboards show outdated data | CloudWatch alarm on job failure; auto-retry; manual DLQ review; dual-write to Redshift as fallback until bake period passes |

## Success Metrics

| Metric | Baseline (Current) | Target (Day 60) |
|--------|-------------------|-----------------|
| Monthly query engine cost | $6,400 (Redshift 24/7) | ≤$1,500 (Athena pay-per-query) |
| Avg dashboard query latency | 8–15 seconds (Redshift, unoptimized) | <10 seconds (Athena on compacted Iceberg gold) |
| Data freshness (dashboards) | 24 hours (nightly Glue→Redshift) | 4 hours (Glue→Iceberg, 6x/day refresh) |
| Self-serve query users | 3 (only engineers comfortable with Redshift) | 25 (all analysts trained on Athena) |